In [1]:
import sys

from copy import deepcopy
from lightning import pytorch as pl
from pathlib import Path

import pandas as pd
import numpy as np
import torch

from dataclasses import InitVar, dataclass
from typing import Sequence
from rdkit import Chem
from rdkit.Chem import Mol, Draw
from rdkit.Chem.rdchem import Atom, Bond, BondType

from chemprop.featurizers.atom import MultiHotAtomFeaturizer
from chemprop.featurizers.bond import MultiHotBondFeaturizer
from chemprop.featurizers.molgraph.molecule import SimpleMoleculeMolGraphFeaturizer

from chemprop.data.molgraph import MolGraph
from chemprop.featurizers.base import GraphFeaturizer
from chemprop.featurizers.molgraph.mixins import _MolGraphFeaturizerMixin

from chemprop import data, featurizers, models

import shap

In [2]:
class CustomMultiHotAtomFeaturizer(MultiHotAtomFeaturizer):
    """A custom MultiHotAtomFeaturizer that allows for selective feature ablation.

    Parameters
    ----------
    keep_features : List[bool], optional
        a list of booleans to indicate which atom features to keep. If None, all features are kept. For any element that is False, the corresponding feature's encoding is set to all zeros. Useful for ablation and SHAP analysis.
    """

    def __init__(self,
                 atomic_nums: Sequence[int],
                 degrees: Sequence[int],
                 formal_charges: Sequence[int],
                 chiral_tags: Sequence[int],
                 num_Hs: Sequence[int],
                 hybridizations: Sequence[int],
                 keep_features: list[bool] | None = None):
        super().__init__(atomic_nums, degrees, formal_charges, chiral_tags, num_Hs, hybridizations)

        if keep_features is None:
            keep_features = [True] * (len(self._subfeats) + 2)
        self.keep_features = keep_features

    def __call__(self, a: Atom | None) -> np.ndarray:
        x = np.zeros(self._MultiHotAtomFeaturizer__size)
        if a is None:
            return x

        feats = [
            a.GetAtomicNum(),
            a.GetTotalDegree(),
            a.GetFormalCharge(),
            int(a.GetChiralTag()),
            int(a.GetTotalNumHs()),
            a.GetHybridization(),
        ]

        i = 0
        for feat, choices, keep in zip(feats, self._subfeats, self.keep_features[:len(feats)]):
            j = choices.get(feat, len(choices))
            if keep:
                x[i + j] = 1
            i += len(choices) + 1

        if self.keep_features[len(feats)]:
            x[i] = int(a.GetIsAromatic())
        if self.keep_features[len(feats) + 1]:
            x[i + 1] = 0.01 * a.GetMass()

        return x

    def zero_mask(self) -> np.ndarray:
        """Featurize the atom by setting all bits to zero."""
        return np.zeros(len(self))

In [3]:
# Example usage
atomic_nums = [6, 7, 8]
degrees = [1, 2, 3]
formal_charges = [-1, 0, 1]
chiral_tags = [0, 1, 2]
num_Hs = [0, 1, 2]
hybridizations = [1, 2, 3]

keep_features_all = [True] * 8
keep_features_some = [True, True, False, True, False, True, True, False]
keep_features_none = [False] * 8

featurizer_all = CustomMultiHotAtomFeaturizer(
    atomic_nums=atomic_nums,
    degrees=degrees,
    formal_charges=formal_charges,
    chiral_tags=chiral_tags,
    num_Hs=num_Hs,
    hybridizations=hybridizations,
    keep_features=keep_features_all
)

featurizer_some = CustomMultiHotAtomFeaturizer(
    atomic_nums=atomic_nums,
    degrees=degrees,
    formal_charges=formal_charges,
    chiral_tags=chiral_tags,
    num_Hs=num_Hs,
    hybridizations=hybridizations,
    keep_features=keep_features_some
)

featurizer_none = CustomMultiHotAtomFeaturizer(
    atomic_nums=atomic_nums,
    degrees=degrees,
    formal_charges=formal_charges,
    chiral_tags=chiral_tags,
    num_Hs=num_Hs,
    hybridizations=hybridizations,
    keep_features=keep_features_none
)

mol = Chem.MolFromSmiles('CCO')
atom = mol.GetAtomWithIdx(0)  # Get the first atom

features = featurizer_all(atom)
print("Atom features all:", features)

features = featurizer_some(atom)
print("Atom features some:", features)

features = featurizer_none(atom)
print("Atom features none:", features)

Atom features all: [1.      0.      0.      0.      0.      0.      0.      1.      0.
 1.      0.      0.      1.      0.      0.      0.      0.      0.
 0.      1.      0.      0.      0.      1.      0.      0.12011]
Atom features some: [1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.
 0. 0.]
Atom features none: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0.]


In [7]:
data = torch.load("../data/folds_no_added/outer_fold_0.pt", weights_only=False)

In [ ]:
data["outer_test"]

SimpleMoleculeMolGraphFeaturizer(atom_featurizer=<chemprop.featurizers.atom.MultiHotAtomFeaturizer object at 0x000001E890F4A6F0>, bond_featurizer=<chemprop.featurizers.bond.MultiHotBondFeaturizer object at 0x000001E890F4A720>, extra_atom_fdim=9, extra_bond_fdim=0)

In [18]:
atomic_features: list[str] = [
    "atomic_fukui_minus",
    "atomic_fukui_plus",
    "atomic_dipole_norm",
    "atomic_quadrupole_principal_invariant_2",
    "atomic_quadrupole_principal_invariant_3",
    "atomic_polarizability_mean",
    "atomic_polarizability_anisotropy",
    "atomic_sasa",
    "partial_charge",
]

In [27]:
from ml_enhance import QuantumFPFileLoader
from collections.abc import Generator


def stream_conformer_df(
    file: Path,
    loader: QuantumFPFileLoader,
) -> Generator[pd.DataFrame, None, None]:
    for df in loader.stream_conformer_dataframe(file):
        df["gibbs_free_energy_300K"] = df["gibbs_free_energy"].map(lambda x: x[1][1])
        yield df

In [28]:
from scipy.constants import (
    Avogadro,  # 1/mol
    Boltzmann,  # in J/K
)

def boltzmann_weights(G: np.ndarray, T: float = 300.0) -> np.ndarray:
    k_B: float = Boltzmann * Avogadro * 0.000239005736
    delta_G = G - G.min()
    factors = np.exp(-delta_G / (k_B * T))
    return factors / factors.sum()

In [44]:
def extract_atom_features(df: pd.DataFrame) -> pd.DataFrame:
    selected_df = df[["original_smiles", "gibbs_free_energy_300K", *atomic_features]]
    object_cols = selected_df.select_dtypes(include="object").columns.to_list()
    exploded_df = selected_df.explode(object_cols)

    G = selected_df["gibbs_free_energy_300K"].unique()
    weights = boltzmann_weights(G)

    arr = np.array(exploded_df[object_cols].values.tolist())  # (n_conformers * n_atoms, n_features, 2)
    atom_idx = arr[:, 0, 0].astype(int)
    values = arr[:, :, 1].astype(float)

    n_conformers = len(weights)
    n_atoms = len(np.unique(atom_idx))
    n_features = len(object_cols)

    atom_matrix = values.reshape(n_conformers, n_atoms, n_features)
    averages = np.einsum("i,ijk->jk", weights, atom_matrix)

    result = pd.DataFrame(averages, columns=object_cols)
    result.insert(0, "atom", np.unique(atom_idx))
    result.insert(0, "original_smiles", selected_df["original_smiles"].iloc[0])

    return result

In [54]:
from IPython.display import display

loader = QuantumFPFileLoader(Path("../data/QuantumFP/QFP_output"))
filelist = loader.list_output_files()

l = []
for file in filelist[:3]:
    for df in stream_conformer_df(file, loader):
        l.append(extract_atom_features(df))

pd.concat(l, ignore_index=True)

,original_smiles,atom,atomic_fukui_minus,atomic_fukui_plus,atomic_dipole_norm,atomic_quadrupole_principal_invariant_2,atomic_quadrupole_principal_invariant_3,atomic_polarizability_mean,atomic_polarizability_anisotropy,atomic_sasa,partial_charge
0,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,1,-0.131132,-0.139724,0.188971,-0.033083,-0.002100,-0.988836,1.755119,83.389156,0.403458
1,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,2,-0.040623,-0.012914,0.079754,-0.181360,-0.021951,0.369055,9.276907,35.080186,-0.276499
2,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,3,-0.032720,-0.066537,0.028973,-0.001489,-0.000001,0.043436,3.954169,28.233808,0.161098
3,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,4,-0.038950,-0.044342,0.176277,-0.091973,-0.004518,-0.065612,5.220691,25.166139,-0.066365
4,[O:1]=[C:2]1[N:3]([H:14])[c:4]2[c:5]([H:15])[c...,5,-0.035337,-0.040210,0.015332,-0.177880,-0.028813,-0.138787,9.687700,52.539365,0.050431
...,...,...,...,...,...,...,...,...,...,...,...
62,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,29,-0.017275,-0.034487,0.141069,-0.002424,0.000028,-0.306551,0.224431,21.820543,-0.046551
63,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,30,-0.013579,-0.033616,0.139773,-0.002436,0.000034,-0.259652,0.251506,21.139856,-0.050922
64,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,31,-0.017666,-0.037328,0.139282,-0.002474,0.000034,-0.269444,0.277749,21.537454,-0.048447
65,[C:1]([C:2]([C:3]([H:19])([H:20])[H:21])([C:4]...,32,-0.089219,-0.052222,0.149762,-0.003719,0.000070,-0.243610,0.269535,19.844542,-0.038102
